# Stratified Data Splitting Algorithm (Thesis Section 4.4.1)

Implements the question-level stratified split optimization used to partition 2,550 responses (85 questions x 30 responses) into Train/Val/Test sets.

**Key Design Decisions:**
- **Question-level split** to prevent data leakage (all 30 responses for a question stay together)
- **60/20/20 ratio** → 51 train / 17 val / 17 test questions
- **Random search optimization** (50,000 iterations) to minimize rubric distribution imbalance
- **Balance score** = sum of squared deviations from overall distribution across all rubric score levels

**Sections:**
1. Load & Explore Dataset
2. Overall Distribution (Optimization Target)
3. Balance Score Function
4. Random Search Optimization (50,000 iterations)
5. Validate Optimal Split
6. Distribution Comparison (Train vs Val vs Test)
7. Difficulty Balance Check
8. Off-Topic Balance Check
9. Comparison with Random Split

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
random.seed(42)
np.random.seed(42)

RUBRICS = [
    ('Clarity', 'Clarity_Score', 2),
    ('Terminology', 'Terminology_Score', 2),
    ('Coverage', 'Coverage_Score', 2),
    ('Accuracy', 'Accuracy_Score', 4)
]

# Known optimal split (from thesis)
TRAIN_QUESTIONS = [1, 2, 3, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 17, 20, 23, 24,
                   27, 28, 29, 30, 33, 35, 36, 38, 40, 41, 43, 44, 46, 47, 49,
                   52, 53, 56, 58, 59, 61, 62, 64, 65, 67, 70, 73, 74, 76, 77,
                   79, 82, 83, 85]

VAL_QUESTIONS = [16, 26, 32, 34, 37, 39, 42, 48, 50, 51, 54, 60, 63, 66, 71, 72, 78]

TEST_QUESTIONS = [4, 8, 18, 19, 21, 22, 25, 31, 45, 55, 57, 68, 69, 75, 80, 81, 84]

## 1. Load & Explore Dataset

In [ ]:
df = pd.read_excel('../Data Statsitical Analysis/Data.xlsx')
df['Q_num'] = df['ID'].str.extract(r'Q(\d+)')[0].astype(int)

n_questions = df['Q_num'].nunique()
n_per_q = df.groupby('Q_num').size()

print(f'Total samples:    {len(df)}')
print(f'Total questions:  {n_questions}')
print(f'Responses per Q:  {n_per_q.min()}-{n_per_q.max()} (all {n_per_q.mode().iloc[0]})')
print(f'\nSplit ratio: 60/20/20 → {51} train / {17} val / {17} test questions')
print(f'                     → {51*30} train / {17*30} val / {17*30} test samples')

## 2. Overall Distribution (Optimization Target)

In [ ]:
def get_overall_distribution(df):
    """Calculate target distribution from full dataset."""
    overall_dist = {}
    for rubric, col, max_score in RUBRICS:
        for score in range(max_score + 1):
            pct = 100 * (df[col] == score).sum() / len(df)
            overall_dist[(rubric, score)] = pct
    return overall_dist


overall_dist = get_overall_distribution(df)

print('Overall Dataset Distribution (optimization target):')
print(f'{"Rubric":<15} {"Score":<8} {"Count":<8} {"Percentage":<10}')
print('-' * 42)
for rubric, col, max_score in RUBRICS:
    for score in range(max_score + 1):
        count = (df[col] == score).sum()
        pct = overall_dist[(rubric, score)]
        print(f'{rubric:<15} {score:<8} {count:<8} {pct:.1f}%')
    print()

In [ ]:
# Visualize overall distribution
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for i, (rubric, col, max_score) in enumerate(RUBRICS):
    ax = axes[i]
    counts = [((df[col] == s).sum()) for s in range(max_score + 1)]
    pcts = [c / len(df) * 100 for c in counts]
    bars = ax.bar(range(max_score + 1), pcts, color=colors[i], edgecolor='white', alpha=0.85)
    ax.set_title(f'{rubric} (0-{max_score})', fontsize=12)
    ax.set_xlabel('Score')
    ax.set_ylabel('% of Responses')
    ax.set_xticks(range(max_score + 1))
    for bar, pct in zip(bars, pcts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{pct:.1f}%', ha='center', fontsize=9)

fig.suptitle('Overall Rubric Score Distribution (N=2,550)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Balance Score Function

In [ ]:
def calculate_balance_score(questions, df, overall_dist):
    """
    Calculate how balanced a set of questions is.
    Lower score = better balance (closer to overall distribution).
    
    Uses sum of squared differences between subset and overall 
    distribution across all rubric score levels.
    """
    subset = df[df['Q_num'].isin(questions)]
    if len(subset) == 0:
        return float('inf')

    total_diff = 0
    for rubric, col, max_score in RUBRICS:
        for score in range(max_score + 1):
            actual_pct = 100 * (subset[col] == score).sum() / len(subset)
            target_pct = overall_dist[(rubric, score)]
            total_diff += (actual_pct - target_pct) ** 2

    return total_diff


# Demo: score of a random split vs the optimal split
random_qs = random.sample(range(1, 86), 51)
random_score = calculate_balance_score(random_qs, df, overall_dist)
optimal_score = calculate_balance_score(TRAIN_QUESTIONS, df, overall_dist)

print(f'Balance score (lower = better):')
print(f'  Random split:  {random_score:.2f}')
print(f'  Optimal split: {optimal_score:.2f}')
print(f'  Improvement:   {random_score / optimal_score:.1f}x better')

## 4. Random Search Optimization (50,000 iterations)

In [ ]:
N_ITERATIONS = 50_000
all_questions = list(range(1, 86))

best_score = float('inf')
best_split = None
score_history = []  # Track convergence
all_scores = []     # All scores for comparison

print(f'Running optimization ({N_ITERATIONS:,} iterations)...')
for i in range(N_ITERATIONS):
    random.shuffle(all_questions)
    train_q = sorted(all_questions[:51])
    val_q = sorted(all_questions[51:68])
    test_q = sorted(all_questions[68:])

    score = calculate_balance_score(train_q, df, overall_dist)
    all_scores.append(score)

    if score < best_score:
        best_score = score
        best_split = {'train': train_q, 'val': val_q, 'test': test_q}
        score_history.append((i, best_score))

    if (i + 1) % 10000 == 0:
        print(f'  {i+1:,} iterations — best score: {best_score:.2f}')

print(f'\nOptimization complete!')
print(f'  Best balance score: {best_score:.2f}')
print(f'  Improvements found: {len(score_history)}')

In [ ]:
# Convergence plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Convergence of best score
ax = axes[0]
iters, scores = zip(*score_history)
ax.plot(iters, scores, 'o-', color='#2ecc71', markersize=5)
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('Best Balance Score', fontsize=11)
ax.set_title('(a) Optimization Convergence', fontsize=12, fontweight='bold')
ax.axhline(best_score, color='red', linestyle='--', alpha=0.5, label=f'Final: {best_score:.2f}')
ax.legend()

# Panel B: Distribution of all tested scores
ax = axes[1]
ax.hist(all_scores, bins=50, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(best_score, color='red', linewidth=2, linestyle='--', label=f'Best: {best_score:.2f}')
ax.axvline(np.median(all_scores), color='orange', linewidth=1.5, linestyle=':', label=f'Median: {np.median(all_scores):.1f}')
ax.set_xlabel('Balance Score', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('(b) Distribution of Tested Configurations', fontsize=12, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Score statistics across {N_ITERATIONS:,} random splits:')
print(f'  Median: {np.median(all_scores):.2f}')
print(f'  Mean:   {np.mean(all_scores):.2f}')
print(f'  Min:    {min(all_scores):.2f}')
print(f'  Max:    {max(all_scores):.2f}')

## 5. Validate Optimal Split

Verify the found split matches the thesis split (or is comparably balanced).

In [ ]:
# Use the known thesis split for validation
thesis_score = calculate_balance_score(TRAIN_QUESTIONS, df, overall_dist)
found_score = calculate_balance_score(best_split['train'], df, overall_dist)

print(f'Thesis split balance score:  {thesis_score:.2f}')
print(f'Found split balance score:   {found_score:.2f}')
print()

# Verify sets
print(f'Thesis Train ({len(TRAIN_QUESTIONS)} questions):')
print(f'  {TRAIN_QUESTIONS}')
print(f'\nThesis Validation ({len(VAL_QUESTIONS)} questions):')
print(f'  {VAL_QUESTIONS}')
print(f'\nThesis Test ({len(TEST_QUESTIONS)} questions):')
print(f'  {TEST_QUESTIONS}')

# Verify no overlap
all_q = set(TRAIN_QUESTIONS) | set(VAL_QUESTIONS) | set(TEST_QUESTIONS)
overlap_tv = set(TRAIN_QUESTIONS) & set(VAL_QUESTIONS)
overlap_tt = set(TRAIN_QUESTIONS) & set(TEST_QUESTIONS)
overlap_vt = set(VAL_QUESTIONS) & set(TEST_QUESTIONS)

print(f'\nIntegrity checks:')
print(f'  Total questions covered: {len(all_q)} (expected 85)')
print(f'  Train-Val overlap:  {len(overlap_tv)} (expected 0)')
print(f'  Train-Test overlap: {len(overlap_tt)} (expected 0)')
print(f'  Val-Test overlap:   {len(overlap_vt)} (expected 0)')
print(f'  All checks passed:  {len(all_q) == 85 and len(overlap_tv) == 0 and len(overlap_tt) == 0 and len(overlap_vt) == 0}')

## 6. Distribution Comparison (Train vs Val vs Test)

In [ ]:
# Compute distributions for each set
train_df = df[df['Q_num'].isin(TRAIN_QUESTIONS)]
val_df = df[df['Q_num'].isin(VAL_QUESTIONS)]
test_df = df[df['Q_num'].isin(TEST_QUESTIONS)]

print(f'Set sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}')
print()

rows = []
for rubric, col, max_score in RUBRICS:
    for score in range(max_score + 1):
        overall_pct = overall_dist[(rubric, score)]
        train_pct = 100 * (train_df[col] == score).sum() / len(train_df)
        val_pct = 100 * (val_df[col] == score).sum() / len(val_df)
        test_pct = 100 * (test_df[col] == score).sum() / len(test_df)

        rows.append({
            'Rubric': rubric, 'Score': score,
            'Overall (%)': round(overall_pct, 1),
            'Train (%)': round(train_pct, 1),
            'Train Diff': round(train_pct - overall_pct, 1),
            'Val (%)': round(val_pct, 1),
            'Val Diff': round(val_pct - overall_pct, 1),
            'Test (%)': round(test_pct, 1),
            'Test Diff': round(test_pct - overall_pct, 1),
        })

dist_df = pd.DataFrame(rows)
print('Distribution Comparison:')
dist_df

In [ ]:
# Deviation heatmap
deviation_data = []
labels = []

for rubric, col, max_score in RUBRICS:
    for score in range(max_score + 1):
        overall_pct = overall_dist[(rubric, score)]
        train_pct = 100 * (train_df[col] == score).sum() / len(train_df)
        val_pct = 100 * (val_df[col] == score).sum() / len(val_df)
        test_pct = 100 * (test_df[col] == score).sum() / len(test_df)
        deviation_data.append([train_pct - overall_pct, val_pct - overall_pct, test_pct - overall_pct])
        labels.append(f'{rubric} ({score})')

dev_matrix = np.array(deviation_data)

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(dev_matrix, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
            xticklabels=['Train', 'Validation', 'Test'],
            yticklabels=labels, ax=ax,
            vmin=-5, vmax=5,
            cbar_kws={'label': 'Deviation from Overall (%)'})
ax.set_title('Distribution Deviation from Overall (percentage points)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Max |deviation| — Train: {np.max(np.abs(dev_matrix[:, 0])):.1f}%')
print(f'Max |deviation| — Val:   {np.max(np.abs(dev_matrix[:, 1])):.1f}%')
print(f'Max |deviation| — Test:  {np.max(np.abs(dev_matrix[:, 2])):.1f}%')

In [ ]:
# Side-by-side bar charts per rubric
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
set_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for i, (rubric, col, max_score) in enumerate(RUBRICS):
    ax = axes[i]
    scores = range(max_score + 1)
    x = np.arange(len(scores))
    width = 0.2

    overall_pcts = [overall_dist[(rubric, s)] for s in scores]
    train_pcts = [100 * (train_df[col] == s).sum() / len(train_df) for s in scores]
    val_pcts = [100 * (val_df[col] == s).sum() / len(val_df) for s in scores]
    test_pcts = [100 * (test_df[col] == s).sum() / len(test_df) for s in scores]

    ax.bar(x - 1.5*width, overall_pcts, width, label='Overall', color='#2c3e50', alpha=0.7)
    ax.bar(x - 0.5*width, train_pcts, width, label='Train', color='#3498db')
    ax.bar(x + 0.5*width, val_pcts, width, label='Val', color='#2ecc71')
    ax.bar(x + 1.5*width, test_pcts, width, label='Test', color='#e74c3c')

    ax.set_xticks(x)
    ax.set_xticklabels([str(s) for s in scores])
    ax.set_xlabel('Score')
    ax.set_ylabel('% of Responses')
    ax.set_title(f'{rubric} (0-{max_score})', fontsize=12, fontweight='bold')
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle('Score Distribution by Split', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Difficulty Balance Check

In [ ]:
# Compute average difficulty (Rubric_Grade) per question
q_difficulty = df.groupby('Q_num')['Rubric_Grade'].mean()

overall_diff = q_difficulty.mean()
train_diff = q_difficulty[q_difficulty.index.isin(TRAIN_QUESTIONS)].mean()
val_diff = q_difficulty[q_difficulty.index.isin(VAL_QUESTIONS)].mean()
test_diff = q_difficulty[q_difficulty.index.isin(TEST_QUESTIONS)].mean()

diff_table = pd.DataFrame({
    'Set': ['Overall', 'Train', 'Validation', 'Test'],
    'N Questions': [85, 51, 17, 17],
    'N Samples': [2550, 1530, 510, 510],
    'Avg Difficulty': [round(overall_diff, 2), round(train_diff, 2),
                       round(val_diff, 2), round(test_diff, 2)],
    'Diff from Overall': [0, round(train_diff - overall_diff, 2),
                          round(val_diff - overall_diff, 2),
                          round(test_diff - overall_diff, 2)]
})
print('Difficulty Balance:')
diff_table

In [ ]:
# Difficulty distribution by set
fig, ax = plt.subplots(figsize=(10, 5))

train_diffs = q_difficulty[q_difficulty.index.isin(TRAIN_QUESTIONS)].values
val_diffs = q_difficulty[q_difficulty.index.isin(VAL_QUESTIONS)].values
test_diffs = q_difficulty[q_difficulty.index.isin(TEST_QUESTIONS)].values

bp = ax.boxplot([train_diffs, val_diffs, test_diffs],
                labels=['Train (51 Q)', 'Validation (17 Q)', 'Test (17 Q)'],
                patch_artist=True, widths=0.5)

box_colors = ['#3498db', '#2ecc71', '#e74c3c']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.axhline(overall_diff, color='black', linestyle='--', alpha=0.5, label=f'Overall mean: {overall_diff:.2f}')
ax.set_ylabel('Mean Question Difficulty (Rubric Grade)', fontsize=11)
ax.set_title('Question Difficulty Distribution by Split', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Off-Topic Balance Check

In [ ]:
# Check off-topic distribution across sets
df['is_off_topic'] = df['off_topic'].isin(['Off-Topic', 'No Answer'])

overall_ot = df['is_off_topic'].mean() * 100
train_ot = train_df['off_topic'].isin(['Off-Topic', 'No Answer']).mean() * 100
val_ot = val_df['off_topic'].isin(['Off-Topic', 'No Answer']).mean() * 100
test_ot = test_df['off_topic'].isin(['Off-Topic', 'No Answer']).mean() * 100

ot_table = pd.DataFrame({
    'Set': ['Overall', 'Train', 'Validation', 'Test'],
    'Off-Topic Count': [
        df['is_off_topic'].sum(),
        train_df['off_topic'].isin(['Off-Topic', 'No Answer']).sum(),
        val_df['off_topic'].isin(['Off-Topic', 'No Answer']).sum(),
        test_df['off_topic'].isin(['Off-Topic', 'No Answer']).sum()
    ],
    'Off-Topic %': [round(overall_ot, 1), round(train_ot, 1),
                    round(val_ot, 1), round(test_ot, 1)],
    'Diff from Overall': [0, round(train_ot - overall_ot, 1),
                          round(val_ot - overall_ot, 1),
                          round(test_ot - overall_ot, 1)]
})
print('Off-Topic Distribution:')
ot_table

## 9. Comparison with Random Split

In [ ]:
# Generate 1000 random splits and compare
random_scores = []
random_max_devs = []

for _ in range(1000):
    qs = list(range(1, 86))
    random.shuffle(qs)
    train_q = qs[:51]

    score = calculate_balance_score(train_q, df, overall_dist)
    random_scores.append(score)

    # Max deviation
    subset = df[df['Q_num'].isin(train_q)]
    max_dev = 0
    for rubric, col, max_score in RUBRICS:
        for s in range(max_score + 1):
            actual = 100 * (subset[col] == s).sum() / len(subset)
            target = overall_dist[(rubric, s)]
            max_dev = max(max_dev, abs(actual - target))
    random_max_devs.append(max_dev)

# Optimal split stats
opt_max_dev = 0
for rubric, col, max_score in RUBRICS:
    for s in range(max_score + 1):
        actual = 100 * (train_df[col] == s).sum() / len(train_df)
        target = overall_dist[(rubric, s)]
        opt_max_dev = max(opt_max_dev, abs(actual - target))

comp_table = pd.DataFrame({
    'Metric': ['Balance Score', 'Max Deviation (%)', 'Difficulty Bias'],
    'Random (median)': [f'{np.median(random_scores):.1f}',
                        f'{np.median(random_max_devs):.1f}%',
                        'Variable'],
    'Optimized': [f'{thesis_score:.2f}',
                  f'{opt_max_dev:.1f}%',
                  f'±{abs(train_diff - overall_diff):.2f}']
})
print('Random vs Optimized Split:')
comp_table

In [ ]:
# Visualization: optimized vs random
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(random_scores, bins=30, color='#95a5a6', edgecolor='white', alpha=0.7, label='Random splits (1000)')
ax.axvline(thesis_score, color='#2ecc71', linewidth=3, linestyle='-', label=f'Optimized: {thesis_score:.2f}')
ax.set_xlabel('Balance Score', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('(a) Balance Score Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

ax = axes[1]
ax.hist(random_max_devs, bins=30, color='#95a5a6', edgecolor='white', alpha=0.7, label='Random splits (1000)')
ax.axvline(opt_max_dev, color='#2ecc71', linewidth=3, linestyle='-', label=f'Optimized: {opt_max_dev:.1f}%')
ax.set_xlabel('Max Deviation from Overall (%)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('(b) Maximum Distribution Deviation', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

percentile = (np.array(random_scores) > thesis_score).mean() * 100
print(f'The optimized split is better than {percentile:.1f}% of random splits')